In [5]:
import random
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image, ImageOps


@dataclass
class CFG:
    train_root: str = r"C:\Users\Mikko\RODI-DATA\Train"
    img_size: int = 50
    batch_size: int = 128
    epochs: int = 6
    lr: float = 1e-3
    seed: int = 42
    num_workers: int = 0
    val_frac: float = 0.2
    neg_samples: int = 2000   # 2000..10000

cfg = CFG()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))


class SquarePad:
    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        m = max(w, h)
        hp = (m - w) // 2
        vp = (m - h) // 2
        padding = (hp, vp, m - w - hp, m - h - vp)
        return ImageOps.expand(img, padding, fill=0)


train_tf = transforms.Compose([
    SquarePad(),
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

val_tf = transforms.Compose([
    SquarePad(),
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

full_train = datasets.ImageFolder(cfg.train_root, transform=train_tf)
full_val   = datasets.ImageFolder(cfg.train_root, transform=val_tf)

print("Classes:", full_train.class_to_idx)
fish_idx = full_train.class_to_idx["fish"]
targets = full_train.targets

# ---- Build subset: all fish + N negatives ----
fish_indices = [i for i, y in enumerate(targets) if y == fish_idx]
neg_indices  = [i for i, y in enumerate(targets) if y != fish_idx]

random.shuffle(neg_indices)
neg_indices = neg_indices[:cfg.neg_samples]

subset_indices = fish_indices + neg_indices
random.shuffle(subset_indices)

print(f"Subset size: {len(subset_indices)} | fish={len(fish_indices)} | neg_used={len(neg_indices)}")

# ---- Stratified split subset ----
def stratified_split_on_subset(indices: List[int], val_frac: float) -> Tuple[List[int], List[int]]:
    fish = [i for i in indices if targets[i] == fish_idx]
    neg  = [i for i in indices if targets[i] != fish_idx]
    random.shuffle(fish)
    random.shuffle(neg)
    n_fish_val = max(1, int(len(fish) * val_frac))
    n_neg_val  = max(1, int(len(neg)  * val_frac))
    val_idx = fish[:n_fish_val] + neg[:n_neg_val]
    train_idx = fish[n_fish_val:] + neg[n_neg_val:]
    random.shuffle(train_idx)
    random.shuffle(val_idx)
    return train_idx, val_idx

train_idx, val_idx = stratified_split_on_subset(subset_indices, cfg.val_frac)

train_ds = Subset(full_train, train_idx)
val_ds   = Subset(full_val, val_idx)

train_fish = sum(1 for i in train_idx if targets[i] == fish_idx)
train_neg  = len(train_idx) - train_fish
val_fish   = sum(1 for i in val_idx if targets[i] == fish_idx)
val_neg    = len(val_idx) - val_fish
print(f"Train: {len(train_idx)} (fish {train_fish}, neg {train_neg})")
print(f"Val  : {len(val_idx)} (fish {val_fish}, neg {val_neg})")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=(device=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=(device=="cuda"))


class SimpleCNN(nn.Module):
    def __init__(self, img_size: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        feat_hw = img_size // 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * feat_hw * feat_hw, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)  # 1 logit for BCE
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

model = SimpleCNN(cfg.img_size).to(device)

# BCEWithLogits: optional pos_weight helps a bit when still imbalanced
pos_weight = torch.tensor([train_neg / max(1, train_fish)], device=device, dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)

@torch.no_grad()
def evaluate():
    model.eval()
    tp = fp = fn = tn = 0
    for x, y_multi in val_loader:
        x = x.to(device, non_blocking=True)
        y_true = (y_multi == fish_idx).to(device, non_blocking=True).to(torch.bool)
        logits = model(x)
        y_pred = (logits >= 0).to(torch.bool)  # threshold 0 = p>=0.5
        tp += (y_pred & y_true).sum().item()
        fp += (y_pred & ~y_true).sum().item()
        fn += (~y_pred & y_true).sum().item()
        tn += (~y_pred & ~y_true).sum().item()

    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn + 1e-9)
    return acc, prec, rec, f1, (tp, fp, fn, tn)

print("\nStarting training...")
for epoch in range(1, cfg.epochs + 1):
    model.train()
    running = 0.0
    for x, y_multi in train_loader:
        x = x.to(device, non_blocking=True)
        y = (y_multi == fish_idx).float().to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item() * x.size(0)

    acc, prec, rec, f1, (tp, fp, fn, tn) = evaluate()
    print(
        f"Epoch {epoch:02d} | train_loss={running/len(train_ds):.4f} | "
        f"val_acc={acc*100:.1f}% val_prec={prec*100:.1f}% val_rec={rec*100:.1f}% val_f1={f1*100:.1f}% | "
        f"TP={tp} FP={fp} FN={fn} TN={tn}"
    )

print("Done.")

DEVICE: cuda
Classes: {'artifact': 0, 'fish': 1, 'insect': 2, 'leave': 3, 'partial-overlapping-objects': 4, 'rounded': 5, 'shadow': 6, 'variable': 7, 'wormish': 8}
Subset size: 2357 | fish=357 | neg_used=2000
Train: 1886 (fish 286, neg 1600)
Val  : 471 (fish 71, neg 400)

Starting training...
Epoch 01 | train_loss=0.9566 | val_acc=82.0% val_prec=44.0% val_rec=71.8% val_f1=54.5% | TP=51 FP=65 FN=20 TN=335
Epoch 02 | train_loss=0.9440 | val_acc=68.4% val_prec=30.7% val_rec=87.3% val_f1=45.4% | TP=62 FP=140 FN=9 TN=260
Epoch 03 | train_loss=0.8491 | val_acc=80.7% val_prec=41.7% val_rec=70.4% val_f1=52.4% | TP=50 FP=70 FN=21 TN=330
Epoch 04 | train_loss=0.8370 | val_acc=79.4% val_prec=40.3% val_rec=76.1% val_f1=52.7% | TP=54 FP=80 FN=17 TN=320
Epoch 05 | train_loss=0.7896 | val_acc=79.8% val_prec=40.9% val_rec=76.1% val_f1=53.2% | TP=54 FP=78 FN=17 TN=322
Epoch 06 | train_loss=0.7369 | val_acc=80.3% val_prec=41.8% val_rec=78.9% val_f1=54.6% | TP=56 FP=78 FN=15 TN=322
Done.
